# 12.3 statsmodels介绍

## 12.3.1 对线性模型进行估计

In [1]:
import statsmodels.api as sm  # 基于数组
import statsmodels.formula.api as smf  # 基于公式
import numpy as np
import pandas as pd
rng = np.random.default_rng(seed=12345)
def dnorm(mean, variance, size=1):
    if isinstance(size, int):
        size = size,
        return mean + np.sqrt(variance) * rng.standard_normal(*size)

# 直接用np.random.normal(mean, std, size)即可

N = 100
X = np.c_[dnorm(0,0.4,size=N),dnorm(0,0.6,size=N),dnorm(0,0.2,size=N)]
eps = dnorm(0,0.1,size=N)
beta = [0.1,0.3,0.5]
y = np.dot(X,beta) + eps
X[:5]

array([[-0.90050602, -0.18942958, -1.0278702 ],
       [ 0.79925205, -1.54598388, -0.32739708],
       [-0.55065483, -0.12025429,  0.32935899],
       [-0.16391555,  0.82403985,  0.20827485],
       [-0.04765129, -0.21314698, -0.04824364]])

In [2]:
y[:5]

array([-0.59952668, -0.58845445,  0.18563386, -0.00747657, -0.01537445])

In [4]:
X_model = sm.add_constant(X)
X_model[:5]

array([[ 1.        , -0.90050602, -0.18942958, -1.0278702 ],
       [ 1.        ,  0.79925205, -1.54598388, -0.32739708],
       [ 1.        , -0.55065483, -0.12025429,  0.32935899],
       [ 1.        , -0.16391555,  0.82403985,  0.20827485],
       [ 1.        , -0.04765129, -0.21314698, -0.04824364]])

In [5]:
model = sm.OLS(y, X)
results = model.fit()
results.params

array([0.06681503, 0.26803235, 0.45052319])

In [6]:
print(results.summary())

                                 OLS Regression Results                                
Dep. Variable:                      y   R-squared (uncentered):                   0.469
Model:                            OLS   Adj. R-squared (uncentered):              0.452
Method:                 Least Squares   F-statistic:                              28.51
Date:                Sat, 04 Apr 2026   Prob (F-statistic):                    2.66e-13
Time:                        14:43:40   Log-Likelihood:                         -25.611
No. Observations:                 100   AIC:                                      57.22
Df Residuals:                      97   BIC:                                      65.04
Df Model:                           3                                                  
Covariance Type:            nonrobust                                                  
                 coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------

In [7]:
data = pd.DataFrame(X, columns=['col0', 'col1', 'col2'])
data['y'] = y
data.head()

,col0,col1,col2,y
0,-0.900506,-0.189430,-1.027870,-0.599527
1,0.799252,-1.545984,-0.327397,-0.588454
2,-0.550655,-0.120254,0.329359,0.185634
3,-0.163916,0.824040,0.208275,-0.007477
4,-0.047651,-0.213147,-0.048244,-0.015374


In [8]:
results = smf.ols('y ~ col0+col1+col2',data=data).fit()
results.params

Intercept   -0.020799
col0         0.065813
col1         0.268970
col2         0.449419
dtype: float64

In [9]:
results.tvalues

Intercept   -0.652501
col0         1.219768
col1         6.312369
col2         6.567428
dtype: float64

In [10]:
results.predict(data[:5])

0   -0.592959
1   -0.531160
2    0.058636
3    0.283658
4   -0.102947
dtype: float64

## 12.3.2 对时间序列过程进行估计

In [11]:
init_x = 4
values = [init_x,init_x]
N = 1000
b0 = 0.8
b1 = -0.4
noise = dnorm(0,0.1,N)
for i in range(N):
    new_x = values[-1]*b0 + values[-2]*b1 + noise[i]
    values.append(new_x)
from statsmodels.tsa.ar_model import AutoReg
MAXLAGS = 5
model = AutoReg(values, lags=MAXLAGS)
results = model.fit()
print(results.summary())

                            AutoReg Model Results                             
Dep. Variable:                      y   No. Observations:                 1002
Model:                     AutoReg(5)   Log Likelihood                -292.347
Method:               Conditional MLE   S.D. of innovations              0.324
Date:                Sat, 04 Apr 2026   AIC                            598.693
Time:                        14:53:05   BIC                            633.026
Sample:                             5   HQIC                           611.744
                                 1002                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.0235      0.010      2.258      0.024       0.003       0.044
y.L1           0.8097      0.032     25.611      0.000       0.748       0.872
y.L2          -0.4287      0.041    -10.540      0.0

# End